# 104. Maximum Depth of Binary Tree

**Easy**

Given the `root` of a binary tree, return *its maximum depth*.

A binary tree's **maximum depth** is the number of nodes along the longest path from the root node down to the farthest leaf node.

---

**Example 1:**

```
        3
       / \
      9   20
         /  \
        15   7

Input:  root = [3, 9, 20, null, null, 15, 7]
Output: 3
```

**Example 2:**

```
Input:  root = [1, null, 2]
Output: 2
```

---

**Constraints:**

- The number of nodes in the tree is in the range `[0, 10^4]`.
- `-100 <= Node.val <= 100`

**Note:** an empty tree (`root = None`) has depth `0`.


### The node definition

LeetCode gives every tree problem this exact class. A tree is just a `root`
node; each node holds a value and points to a left child and a right child
(either can be `None`). Run this cell first.

In [ ]:
# Definition for a binary tree node.
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


### My Queue - built out of a linked list

`Queue` keeps a pointer to the **first** *and* the **last** node, so
`emfilie` (enqueue) and `defilie` (dequeue) are both `O(1)`. A Python list used
as a queue would make `pop(0)` cost `O(n)` because every element shifts down.

`self.size` is updated in both methods, so asking "how many are waiting?" is
`O(1)` too - that is what makes the batch trick possible.

*(This cell is my code, untouched. The only thing removed is a second copy of
`TreeNode` that was already defined in the cell above.)*

In [67]:
class QueueNode:
    def __init__(self, node: TreeNode, next=None):
        self.node = node
        self.next = next


class Queue:
    def __init__(self):
        self.first = None
        self.last = None
        self.size = 0

    def isEmpty(self):
        return self.first is None

    def emfilie(self, node: TreeNode):
        newNode = QueueNode(node)

        if self.first is None:
            self.first = newNode
            self.last = newNode
        else:
            self.last.next = newNode
            self.last = newNode
        self.size += 1

    def defilie(self)->TreeNode:
        if self.first is None:
            return None

        node = self.first.node
        self.first = self.first.next

        if self.first is None:
            self.last = None
        self.size -= 1
        return node
    def printQ(self)->list:
        if self.first is None: return []
        else :
            lst: list =  []
            i = self.first
            for _ in range(self.size):
                lst.append(i.node.val)
                i = i.next
            return lst


### My solutions - all three in one class

Nothing below was rewritten. The only edits are the mechanical ones needed to
live inside a class: `self` as the first parameter, one level of indentation,
a different name per version (three methods called `maxDepth` would shadow each
other), and `maxDepth(...)` -> `self.maxDepthV2(...)` on the recursive calls.

| version | idea | verdict |
|---|---|---|
| **V1** | walk down with `current`, add up two counters | ⚠️ **infinite loop - do not call it** |
| **V2** | recursion: `1 + max(left, right)` | ✅ correct |
| **V3** | BFS with my `Queue`, counting batches | ✅ correct, and survives deep trees |

**Why V1 hangs:** the loop is `while current:`, but `current` only moves when
the node has a child. Reach a leaf and neither `if` fires, so `current` never
changes and the condition stays true forever. On Example 1: `current` becomes
`9`, which has no children, and the loop spins. Worth keeping - the bug is the
lesson.

The `print(...)` lines in V2 are mine and they are kept. Comment them out when
you want clean test output.

In [78]:
class Solution:

    # ---------- V1 : first attempt - INFINITE LOOP, kept for the record ----------
    def maxDepthV1(self, root: TreeNode) -> int:
        if root is None: return 0
        head = root
        leftMax = 1
        rightMax = 1
        current = root
        while current:
            if current.left:
                current = current.left
                leftMax += self.maxDepthV1(current.left)
            if current.right:
                current = current.right
                rightMax += self.maxDepthV1(current.right)
        return leftMax if rightMax < leftMax else rightMax

    # ---------- V2 : recursion ----------
    def maxDepthV2(self, root: TreeNode) -> int:
        if root is None:
            return 0

        left = self.maxDepthV2(root.left)
        print(f"left : {left} rootvalue {root.val}")
        right = self.maxDepthV2(root.right)
        print(f"right : {right} rootvalue {root.val}")

        return 1 + max(left, right)

    # ---------- V3 : BFS, counting batches ----------
    def maxDepthV3(self, root: TreeNode):
        if root == None: return 0
        Q = Queue()
        Q.emfilie(root)
        i: TreeNode = root
        batch = 0
        while not Q.isEmpty():
            size = Q.size
            for _ in range(size):
                i = Q.defilie()
                if i.left:
                    Q.emfilie(i.left)
                if i.right:
                    Q.emfilie(i.right)
            batch += 1
        return batch


### Tests - the two LeetCode examples

In [80]:
sol = Solution()
# 8) lopsided - wide+shallow on the left, thin+deep on the right
#        1
#       / \
#      2   3
#     / \   \
#    4   5   6
#             \
#              7
#               \
#                8
t8 = TreeNode(1,
              TreeNode(2, TreeNode(4), TreeNode(5)),
              TreeNode(3, None,
                TreeNode(6, None,
                  TreeNode(7, None,
                    TreeNode(8)))))
print("V3:", sol.maxDepthV3(t8))    # 3



size = 1  || Q = [[1]]
                    IN THE LOOP                 
                    _ = 0 || i = 1
                    size = 2  || Q = [[2, 3]]
size = 2  || Q = [[2, 3]]
                    IN THE LOOP                 
                    _ = 0 || i = 2
                    size = 3  || Q = [[3, 4, 5]]
                    IN THE LOOP                 
                    _ = 1 || i = 3
                    size = 3  || Q = [[4, 5, 6]]
size = 3  || Q = [[4, 5, 6]]
                    IN THE LOOP                 
                    _ = 0 || i = 4
                    size = 2  || Q = [[5, 6]]
                    IN THE LOOP                 
                    _ = 1 || i = 5
                    size = 1  || Q = [[6]]
                    IN THE LOOP                 
                    _ = 2 || i = 6
                    size = 1  || Q = [[7]]
size = 1  || Q = [[7]]
                    IN THE LOOP                 
                    _ = 0 || i = 7
                    size = 1  || Q 

### More test cases

The two LeetCode examples are gentle. These are the ones that catch bugs. Each
tree is drawn so you can count the answer by eye before running it.

**#7** matters most: the long path is on the *left*. Code that peeks at only one
side, or returns the first branch it finishes instead of the **max** of the two,
passes Examples 1 and 2 and dies here.

In [ ]:
# ---------- more tests : V2 (recursion) vs V3 (BFS) ----------
sol = Solution()

# 3) a single node
t3 = TreeNode(1)

# 4) perfect tree - every leaf sits on the same level
#         1
#       /   \
#      2     3
#     / \   / \
#    4   5 6   7
t4 = TreeNode(1,
              TreeNode(2, TreeNode(4), TreeNode(5)),
              TreeNode(3, TreeNode(6), TreeNode(7)))

# 5) left-skewed chain of 5  (a linked list wearing a tree costume)
t5 = TreeNode(1, TreeNode(2, TreeNode(3, TreeNode(4, TreeNode(5)))))

# 6) right-skewed chain of 5
t6 = TreeNode(1, None,
      TreeNode(2, None,
        TreeNode(3, None,
          TreeNode(4, None,
            TreeNode(5)))))

# 7) the DEEP side is the left one
#        1
#       / \
#      2   3
#     /
#    4
#   /
#  5
t7 = TreeNode(1,
              TreeNode(2, TreeNode(4, TreeNode(5))),
              TreeNode(3))

# 8) lopsided - wide+shallow on the left, thin+deep on the right
#        1
#       / \
#      2   3
#     / \   \
#    4   5   6
#             \
#              7
#               \
#                8
t8 = TreeNode(1,
              TreeNode(2, TreeNode(4), TreeNode(5)),
              TreeNode(3, None,
                TreeNode(6, None,
                  TreeNode(7, None,
                    TreeNode(8)))))

for name, tree, expected in [("t3", t3, 1), ("t4", t4, 3), ("t5", t5, 5),
                             ("t6", t6, 5), ("t7", t7, 4), ("t8", t8, 5)]:
    got = sol.maxDepthV3(tree)
    print(f"{name}: V3 = {got}   expected {expected}   {'OK' if got == expected else 'WRONG'}")


### Stress test - where an unbalanced tree bites

The constraints allow **10^4 nodes**, and nothing says the tree is balanced, so
a chain of 10 000 nodes is a *legal input*.

This is the cell that separates the two solutions. V2 recurses once per level;
V3 loops. Read the error V2 raises - it is not a bug in the logic.

In [ ]:
import sys
print("python's recursion limit:", sys.getrecursionlimit())

# build a 10 000-node left chain, iteratively (no recursion while building)
deep = TreeNode(0)
for _ in range(9999):
    deep = TreeNode(0, deep)

# V3 : BFS - the queue never holds more than 1 node for a chain
print("V3:", sol.maxDepthV3(deep))

# V2 : recursion - 10 000 nested calls against a 1000-frame limit
try:
    print("V2:", sol.maxDepthV2(deep))
except RecursionError as e:
    print("V2 raised RecursionError ->", e)
